# Fase 3 — SRNet como 2º adversário (Colab GPU)

Adiciona a **SRNet** ao laço adversarial, sobre o gerador já treinado da Fase 2.

- Crítico WGAN-GP → **realismo** (já vem da Fase 2)
- SRNet → **indetectabilidade** (novo, treinado do zero)

**Critérios de saída:** acurácia de bits ≥ 0,99 · PSNR ≥ 35,3 dB (Fase 2 − 1 dB) · AUC do laço ≈ 0,5.

> A AUC medida aqui é a do detector que **participou do treino** — ela não prova segurança.
> A prova é a Fase 4, com esteganalisador independente. Ver `ROADMAP.md`.

Notebook autossuficiente: o código é gravado pelas próprias células, sem upload.


## 1. Checar a GPU

In [ ]:
import torch
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print('SEM GPU! Vá em Ambiente de execução → Alterar o tipo de ambiente → GPU e reinicie.')


## 2. Montar o Google Drive

É daqui que vem o checkpoint da Fase 2, e é para cá que os da Fase 3 vão ser espelhados.

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/IC_stego'
except Exception as e:
    print('Drive não montado (', e, ') -> usando /content (TEMPORÁRIO, você perde tudo ao fim da sessão)')
    BASE = '/content/IC_stego'

PASTA_DADOS_BASE = os.path.join(BASE, 'datasets')
PASTA_SAIDA_BASE = os.path.join(BASE, 'runs')
os.makedirs(PASTA_DADOS_BASE, exist_ok=True); os.makedirs(PASTA_SAIDA_BASE, exist_ok=True)
print('Dataset ->', PASTA_DADOS_BASE)
print('Saída   ->', PASTA_SAIDA_BASE)


## 3. Gravar o código (sem upload)

Três arquivos: a Fase 3 importa a Fase 2, que importa a Fase 0.

In [ ]:
%%writefile fase0_codificador_decodificador.py
import os
import glob
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image


CONFIG = {
  
    "pasta_dados": None,
    "canais": 3,               # 1 = grayscale (BOSSBase) | 3 = RGB (DIV2K, COCO)
    "tamanho_img": 128,        
    "fracao_val": 0.05,       
    "max_imagens": None,      
    

    "profundidade_dados": 1,  
    "tamanho_msg": None,      


    "tamanho_lote": 16,       
    "epocas": 30,
    "taxa_aprend": 1e-3,      
    "largura": 32,             
    "lambda_imagem": 1.0,      
    "lambda_mensagem": 1.0,    
    "lambda_residuo": 0.0,    
    "usar_amp": True,          
    "alvo_bitacc": 0.99,       

    # --- infra ---
    "num_trabalhadores": 2,   
    "semente": 42,             
    "pasta_saida": "runs/phase0",
    "salvar_cada": 1,        
    "dispositivo": "cuda" if torch.cuda.is_available() else "cpu",
}



COLAB_GPU = dict(
    tamanho_img=128,
    tamanho_msg=32,         
    tamanho_lote=32,
    epocas=120,
    largura=64,
    max_imagens=None,        # dataset inteiro
    num_trabalhadores=4,
    lambda_imagem=2.0,
    lambda_residuo=0.05,
)


EXTENSOES_IMG = (".png", ".jpg", ".jpeg", ".bmp", ".pgm", ".ppm", ".tif", ".tiff", ".webp")


class DatasetImagens(Dataset):
    def __init__(self, caminhos, canais, tamanho_img, treino=True):
        self.caminhos = caminhos
        self.canais = canais
        self.modo = "L" if canais == 1 else "RGB"   # "L" = grayscale
        # Treino: recorte aleatório + espelhamento (aumenta os dados e desacopla do tamanho original).
        # Se a imagem for menor que o recorte, faz resize antes.
        if treino:
            self.transformacao = transforms.Compose([
                transforms.Resize(tamanho_img),                       # garante lado >= tamanho_img
                transforms.RandomCrop(tamanho_img, pad_if_needed=True),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),                                # pixels em [0, 1]
            ])
        else:
            self.transformacao = transforms.Compose([
                transforms.Resize(tamanho_img),
                transforms.CenterCrop(tamanho_img),
                transforms.ToTensor(),
            ])

    def __len__(self):
        return len(self.caminhos)

    def __getitem__(self, indice):
        imagem = Image.open(self.caminhos[indice]).convert(self.modo)
        return self.transformacao(imagem)


class DatasetSintetico(Dataset):
    def __init__(self, canais, tamanho_img, n=512):
        self.canais, self.tamanho_img, self.n = canais, tamanho_img, n

    def __len__(self):
        return self.n

    def __getitem__(self, indice):
        # imagem suave (não ruído puro) para o MSE ter o que aprender
        base = torch.rand(self.canais, 1, 1)
        gradiente = torch.linspace(0, 1, self.tamanho_img).view(1, 1, -1)
        return (base + 0.3 * gradiente).clamp(0, 1).expand(
            self.canais, self.tamanho_img, self.tamanho_img).clone()


def montar_loaders(cfg):
    if cfg["pasta_dados"]:
        caminhos = []
        for ext in EXTENSOES_IMG:
            caminhos += glob.glob(os.path.join(cfg["pasta_dados"], "**", "*" + ext), recursive=True)
            caminhos += glob.glob(os.path.join(cfg["pasta_dados"], "**", "*" + ext.upper()), recursive=True)
        caminhos = sorted(set(caminhos))
        if not caminhos:
            raise FileNotFoundError(f"Nenhuma imagem encontrada em {cfg['pasta_dados']}")
        random.Random(cfg["semente"]).shuffle(caminhos)
        if cfg.get("max_imagens"):
            caminhos = caminhos[:cfg["max_imagens"]]
            print(f"[dados] max_imagens={cfg['max_imagens']} -> usando {len(caminhos)} imagens")
        n_val = max(1, int(len(caminhos) * cfg["fracao_val"]))
        caminhos_val, caminhos_treino = caminhos[:n_val], caminhos[n_val:]
        print(f"[dados] {len(caminhos_treino)} treino / {len(caminhos_val)} val  ({cfg['pasta_dados']})")
        ds_treino = DatasetImagens(caminhos_treino, cfg["canais"], cfg["tamanho_img"], treino=True)
        ds_val = DatasetImagens(caminhos_val, cfg["canais"], cfg["tamanho_img"], treino=False)
    else:
        print("[dados] pasta_dados=None -> usando dataset SINTÉTICO (teste rápido)")
        ds_treino = DatasetSintetico(cfg["canais"], cfg["tamanho_img"], n=512)
        ds_val = DatasetSintetico(cfg["canais"], cfg["tamanho_img"], n=64)

    loader_treino = DataLoader(ds_treino, batch_size=cfg["tamanho_lote"], shuffle=True,
                               num_workers=cfg["num_trabalhadores"], drop_last=True, pin_memory=True)
    loader_val = DataLoader(ds_val, batch_size=cfg["tamanho_lote"], shuffle=False,
                            num_workers=cfg["num_trabalhadores"], pin_memory=True)
    return loader_treino, loader_val


def bloco_conv(canais_entrada, canais_saida):
    return nn.Sequential(
        nn.Conv2d(canais_entrada, canais_saida, 3, padding=1),
        nn.BatchNorm2d(canais_saida),
        nn.LeakyReLU(0.2, inplace=True),
    )


class Codificador(nn.Module):

    def __init__(self, canais, profundidade_dados, largura=32):
        super().__init__()
        self.bloco1 = bloco_conv(canais, largura)
        self.bloco2 = bloco_conv(largura + profundidade_dados, largura)
        self.bloco3 = bloco_conv(2 * largura + profundidade_dados, largura)
        self.para_residuo = nn.Conv2d(3 * largura + profundidade_dados, canais, 3, padding=1)

    def forward(self, portadora, mensagem):
        a = self.bloco1(portadora)
        b = self.bloco2(torch.cat([a, mensagem], dim=1))
        c = self.bloco3(torch.cat([a, b, mensagem], dim=1))
        residuo = self.para_residuo(torch.cat([a, b, c, mensagem], dim=1))
        estego = (portadora + residuo).clamp(0.0, 1.0)   # soma o resíduo e mantém em [0,1]
        return estego


class Decodificador(nn.Module):
    

    def __init__(self, canais, profundidade_dados, largura=32):
        super().__init__()
        self.bloco1 = bloco_conv(canais, largura)
        self.bloco2 = bloco_conv(largura, largura)
        self.bloco3 = bloco_conv(2 * largura, largura)
        self.para_logits = nn.Conv2d(3 * largura, profundidade_dados, 3, padding=1)

    def forward(self, estego):
        a = self.bloco1(estego)
        b = self.bloco2(a)
        c = self.bloco3(torch.cat([a, b], dim=1))
        return self.para_logits(torch.cat([a, b, c], dim=1))


def _tamanho_msg(cfg):
    return cfg.get("tamanho_msg") or cfg["tamanho_img"]


def mensagem_aleatoria(tamanho_lote, cfg):
    tm = _tamanho_msg(cfg)
    return torch.randint(0, 2, (tamanho_lote, cfg["profundidade_dados"], tm, tm),
                         dtype=torch.float32, device=cfg["dispositivo"])


def espalhar_mensagem(mensagem, cfg):
    if _tamanho_msg(cfg) == cfg["tamanho_img"]:
        return mensagem
    return F.interpolate(mensagem, size=cfg["tamanho_img"], mode="nearest")


def agregar_logits(logits, cfg):
    if _tamanho_msg(cfg) == cfg["tamanho_img"]:
        return logits
    return F.adaptive_avg_pool2d(logits, _tamanho_msg(cfg))


@torch.no_grad()
def acuracia_bits(logits, mensagem):
    predito = (torch.sigmoid(logits) > 0.5).float()
    return (predito == mensagem).float().mean().item()


def _janela_gaussiana(tamanho, sigma, canais, dispositivo):
    coords = torch.arange(tamanho, device=dispositivo) - tamanho // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    janela2d = (g[:, None] * g[None, :])
    return janela2d.expand(canais, 1, tamanho, tamanho).contiguous()


@torch.no_grad()
def ssim(a, b, canais, tamanho_janela=11, sigma=1.5):

    janela = _janela_gaussiana(tamanho_janela, sigma, canais, a.device)
    pad = tamanho_janela // 2
    mu_a = F.conv2d(a, janela, padding=pad, groups=canais)
    mu_b = F.conv2d(b, janela, padding=pad, groups=canais)
    mu_a2, mu_b2, mu_ab = mu_a * mu_a, mu_b * mu_b, mu_a * mu_b
    var_a = F.conv2d(a * a, janela, padding=pad, groups=canais) - mu_a2
    var_b = F.conv2d(b * b, janela, padding=pad, groups=canais) - mu_b2
    cov_ab = F.conv2d(a * b, janela, padding=pad, groups=canais) - mu_ab
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    s = ((2 * mu_ab + c1) * (2 * cov_ab + c2)) / ((mu_a2 + mu_b2 + c1) * (var_a + var_b + c2))
    return s.mean().item()

def treinar():
    cfg = CONFIG
    torch.manual_seed(cfg["semente"]); random.seed(cfg["semente"])
    os.makedirs(cfg["pasta_saida"], exist_ok=True)
    dispositivo = cfg["dispositivo"]
    tm = _tamanho_msg(cfg)
    bpp = cfg["profundidade_dados"] * (tm / cfg["tamanho_img"]) ** 2
    print(f"[infra] dispositivo={dispositivo}  payload={bpp:.4g} bpp  "
          f"(tamanho_msg={tm}, tamanho_img={cfg['tamanho_img']})")

    loader_treino, loader_val = montar_loaders(cfg)

    codificador = Codificador(cfg["canais"], cfg["profundidade_dados"], cfg["largura"]).to(dispositivo)
    decodificador = Decodificador(cfg["canais"], cfg["profundidade_dados"], cfg["largura"]).to(dispositivo)


    otimizador = torch.optim.Adam(
        list(codificador.parameters()) + list(decodificador.parameters()), lr=cfg["taxa_aprend"])
    usar_amp = cfg["usar_amp"] and dispositivo == "cuda"
    escalador = torch.cuda.amp.GradScaler(enabled=usar_amp)

    melhor = {"score": -1.0, "epoca": 0, "acuracia_bits": 0.0, "psnr": 0.0, "ssim": 0.0}

    for epoca in range(1, cfg["epocas"] + 1):
        codificador.train(); decodificador.train()
        acumulado = {"img": 0.0, "msg": 0.0, "acc": 0.0, "n": 0}

        for portadora in loader_treino:
            portadora = portadora.to(dispositivo, non_blocking=True)
            mensagem = mensagem_aleatoria(portadora.size(0), cfg)
            mensagem_espalhada = espalhar_mensagem(mensagem, cfg)

            with torch.cuda.amp.autocast(enabled=usar_amp):
                estego = codificador(portadora, mensagem_espalhada)
                logits = agregar_logits(decodificador(estego), cfg)

                perda_imagem = F.mse_loss(estego, portadora)                              # qualidade visual
                perda_mensagem = F.binary_cross_entropy_with_logits(logits, mensagem)     # recuperação
                perda_residuo = (estego - portadora).abs().mean()                         # perturbação pequena
                perda = (cfg["lambda_imagem"] * perda_imagem
                         + cfg["lambda_mensagem"] * perda_mensagem
                         + cfg["lambda_residuo"] * perda_residuo)


            otimizador.zero_grad()
            escalador.scale(perda).backward()
            escalador.step(otimizador)
            escalador.update()

            n_lote = portadora.size(0)
            acumulado["img"] += perda_imagem.item() * n_lote
            acumulado["msg"] += perda_mensagem.item() * n_lote
            acumulado["acc"] += acuracia_bits(logits, mensagem) * n_lote
            acumulado["n"] += n_lote

        n = acumulado["n"]
        print(f"[treino] época {epoca:3d}  img_mse={acumulado['img']/n:.5f}  "
              f"msg_bce={acumulado['msg']/n:.4f}  acuracia_bits={acumulado['acc']/n:.4f}")

       
        val = avaliar(codificador, decodificador, loader_val, cfg)
        ok = "OK" if (val["acuracia_bits"] >= cfg["alvo_bitacc"] and val["psnr"] >= 35.0) else "  "
        print(f"[  val ] época {epoca:3d}  acuracia_bits={val['acuracia_bits']:.4f}  "
              f"psnr={val['psnr']:.2f}dB  ssim={val['ssim']:.4f}  linf={val['linf']:.4f}  {ok}")

      
        score = (val["psnr"] if val["acuracia_bits"] >= cfg["alvo_bitacc"]
                 else val["acuracia_bits"] - 100.0)
        if score > melhor["score"]:
            melhor = {"score": score, "epoca": epoca,
                      **{k: val[k] for k in ("acuracia_bits", "psnr", "ssim")}}
            torch.save({"codificador": codificador.state_dict(),
                        "decodificador": decodificador.state_dict(),
                        "epoca": epoca, "cfg": cfg, "val": val},
                       os.path.join(cfg["pasta_saida"], "ckpt_best.pt"))

        if epoca % cfg["salvar_cada"] == 0:
            salvar_amostras(codificador, loader_val, cfg, epoca)
            torch.save({"codificador": codificador.state_dict(),
                        "decodificador": decodificador.state_dict(),
                        "epoca": epoca, "cfg": cfg},
                       os.path.join(cfg["pasta_saida"], "ckpt_last.pt"))

    print(f"\n[fim] checkpoints e amostras em: {cfg['pasta_saida']}")
    print(f"[melhor] época {melhor['epoca']}: acuracia_bits={melhor['acuracia_bits']:.4f}  "
          f"psnr={melhor['psnr']:.2f}dB  ssim={melhor['ssim']:.4f}")
    if melhor["acuracia_bits"] >= cfg["alvo_bitacc"] and melhor["psnr"] >= 35.0:
        print("[Fase 0 FECHADA] acurácia e PSNR>=35dB atingidos -> pode ligar a WGAN-GP (Fase 2).")
    else:
        print("[Fase 0 ainda aberta] baixar o payload (tamanho_msg menor) e/ou treinar mais antes da Fase 2.")
    return {"acuracia_bits": melhor["acuracia_bits"], "psnr": melhor["psnr"], "ssim": melhor["ssim"]}


@torch.no_grad()
def avaliar(codificador, decodificador, loader, cfg):
    codificador.eval(); decodificador.eval()
    acc, mse, ss, linf, n = 0.0, 0.0, 0.0, 0.0, 0
    for portadora in loader:
        portadora = portadora.to(cfg["dispositivo"])
        mensagem = mensagem_aleatoria(portadora.size(0), cfg)
        estego = codificador(portadora, espalhar_mensagem(mensagem, cfg))
        logits = agregar_logits(decodificador(estego), cfg)
        n_lote = portadora.size(0)
        acc += acuracia_bits(logits, mensagem) * n_lote
        mse += F.mse_loss(estego, portadora).item() * n_lote
        ss += ssim(estego, portadora, cfg["canais"]) * n_lote
        linf = max(linf, (estego - portadora).abs().max().item())
        n += n_lote
    mse = mse / max(n, 1)
    psnr = 10.0 * math.log10(1.0 / mse) if mse > 0 else float("inf")
    return {"acuracia_bits": acc / max(n, 1), "psnr": psnr, "ssim": ss / max(n, 1), "linf": linf}


@torch.no_grad()
def salvar_amostras(codificador, loader, cfg, epoca):
    codificador.eval()
    portadora = next(iter(loader)).to(cfg["dispositivo"])[:8]
    mensagem = mensagem_aleatoria(portadora.size(0), cfg)
    estego = codificador(portadora, espalhar_mensagem(mensagem, cfg))
    # amplia o resíduo p/ inspeção visual do que foi alterado
    residuo = (estego - portadora).abs()
    residuo = residuo / (residuo.amax() + 1e-8)
    grade = torch.cat([portadora, estego, residuo], dim=0)   # linha1=portadora, 2=estego, 3=resíduo
    caminho = os.path.join(cfg["pasta_saida"], f"samples_epoch{epoca:03d}.png")
    save_image(grade, caminho, nrow=8)


if __name__ == "__main__":
    treinar()


In [ ]:
%%writefile fase2_wgan_gp.py

import os
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

import fase0_codificador_decodificador as base   # reusa arquitetura, loaders e métricas da Fase 0


CONFIG = dict(base.CONFIG)
CONFIG.update({
    "lambda_adv": 0.01,       
                               
    "n_critic": 5,             
    "lambda_gp": 10.0,         
    "largura_critico": 64,     
    "lr_critico": 1e-4,       
    "beta1": 0.0, "beta2": 0.9,  # betas do Adam (recomendado p/ WGAN-GP)

    "taxa_aprend": 1e-4,      
    "epocas": 60,
    "usar_amp": False,         # GP + AMP não combinam
    "carregar_fase0": "runs/phase0_bossbase/ckpt_best.pt",  # warm start; None = do zero
    "pasta_saida": "runs/phase2_bossbase",
})

COLAB_GPU = dict(base.COLAB_GPU)
COLAB_GPU.update({
    "epocas": 60,
    "lambda_adv": 0.01,
    "usar_amp": False,
})


class Critico(nn.Module):

    def __init__(self, canais, largura=64):
        super().__init__()

        def bloco(entrada, saida, norm=True):
            camadas = [nn.Conv2d(entrada, saida, 4, stride=2, padding=1)]  # reduz H,W pela metade
            if norm:
                camadas.append(nn.InstanceNorm2d(saida, affine=True))
            camadas.append(nn.LeakyReLU(0.2, inplace=True))
            return nn.Sequential(*camadas)

        self.corpo = nn.Sequential(
            bloco(canais, largura, norm=False),   # 1ª camada sem norm
            bloco(largura, largura * 2),
            bloco(largura * 2, largura * 4),
            bloco(largura * 4, largura * 8),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)          # vira (B, C, 1, 1) p/ qualquer tamanho de entrada
        self.para_score = nn.Linear(largura * 8, 1)  # score escalar por imagem

    def forward(self, x):
        h = self.corpo(x)
        h = self.pool(h).flatten(1)     # (B, C)
        return self.para_score(h).squeeze(1)  # (B,)


def penalidade_gradiente(critico, reais, falsas, dispositivo):

    b = reais.size(0)
    eps = torch.rand(b, 1, 1, 1, device=dispositivo)
    interp = (eps * reais + (1.0 - eps) * falsas).requires_grad_(True)
    score = critico(interp)
    grads = torch.autograd.grad(
        outputs=score, inputs=interp,
        grad_outputs=torch.ones_like(score),
        create_graph=True, retain_graph=True, only_inputs=True,
    )[0]
    grads = grads.reshape(b, -1)
    return ((grads.norm(2, dim=1) - 1.0) ** 2).mean()


def treinar():
    cfg = CONFIG
    torch.manual_seed(cfg["semente"]); random.seed(cfg["semente"])
    os.makedirs(cfg["pasta_saida"], exist_ok=True)
    dispositivo = cfg["dispositivo"]
    tm = base._tamanho_msg(cfg)
    bpp = cfg["profundidade_dados"] * (tm / cfg["tamanho_img"]) ** 2
    print(f"[infra] dispositivo={dispositivo}  payload={bpp:.4g} bpp  "
          f"lambda_adv={cfg['lambda_adv']}  n_critic={cfg['n_critic']}")

    loader_treino, loader_val = base.montar_loaders(cfg)

    codificador = base.Codificador(cfg["canais"], cfg["profundidade_dados"], cfg["largura"]).to(dispositivo)
    decodificador = base.Decodificador(cfg["canais"], cfg["profundidade_dados"], cfg["largura"]).to(dispositivo)
    critico = Critico(cfg["canais"], cfg["largura_critico"]).to(dispositivo)

    # --- warm start da Fase 0 (colocar o adversário sobre uma base que já funciona) 
    ck = cfg.get("carregar_fase0")
    if ck and os.path.exists(ck):
        estado = torch.load(ck, map_location=dispositivo)
        codificador.load_state_dict(estado["codificador"])
        decodificador.load_state_dict(estado["decodificador"])
        print(f"[warm start] pesos da Fase 0 carregados de {ck}")
    else:
        print(f"[warm start] {ck!r} não encontrado -> treinando do zero "
              "(recomendado rodar a Fase 0 antes)")

    # gerador = codificador + decodificador (treinados juntos); crítico separado
    opt_g = torch.optim.Adam(list(codificador.parameters()) + list(decodificador.parameters()),
                             lr=cfg["taxa_aprend"], betas=(cfg["beta1"], cfg["beta2"]))
    opt_c = torch.optim.Adam(critico.parameters(),
                             lr=cfg["lr_critico"], betas=(cfg["beta1"], cfg["beta2"]))

    melhor = {"score": -1.0, "epoca": 0, "acuracia_bits": 0.0, "psnr": 0.0, "ssim": 0.0}

    for epoca in range(1, cfg["epocas"] + 1):
        codificador.train(); decodificador.train(); critico.train()
        ac = {"c": 0.0, "w": 0.0, "adv": 0.0, "img": 0.0, "msg": 0.0, "acc": 0.0, "n": 0}

        for portadora in loader_treino:
            portadora = portadora.to(dispositivo, non_blocking=True)
            b = portadora.size(0)

            # ============ 1) treina o CRÍTICO n_critic vezes ============
            for _ in range(cfg["n_critic"]):
                mensagem = base.mensagem_aleatoria(b, cfg)
                with torch.no_grad():                       # não propaga p/ o gerador aqui
                    estego = codificador(portadora, base.espalhar_mensagem(mensagem, cfg))
                score_real = critico(portadora).mean()
                score_falsa = critico(estego).mean()
                gp = penalidade_gradiente(critico, portadora, estego, dispositivo)
                # crítico MINIMIZA:  E[C(falsa)] - E[C(real)] + lambda_gp * GP
                perda_c = score_falsa - score_real + cfg["lambda_gp"] * gp
                opt_c.zero_grad(); perda_c.backward(); opt_c.step()

            w_dist = (score_real - score_falsa).item()      # estimativa da distância de Wasserstein

            # ============ 2) treina o GERADOR 1 vez ============
            mensagem = base.mensagem_aleatoria(b, cfg)
            estego = codificador(portadora, base.espalhar_mensagem(mensagem, cfg))
            logits = base.agregar_logits(decodificador(estego), cfg)

            perda_img = F.mse_loss(estego, portadora)
            perda_msg = F.binary_cross_entropy_with_logits(logits, mensagem)
            perda_res = (estego - portadora).abs().mean()
            perda_adv = -critico(estego).mean()             # gerador quer SUBIR o score da estego
            perda_g = (cfg["lambda_imagem"] * perda_img
                       + cfg["lambda_mensagem"] * perda_msg
                       + cfg["lambda_residuo"] * perda_res
                       + cfg["lambda_adv"] * perda_adv)
            opt_g.zero_grad(); perda_g.backward(); opt_g.step()

            ac["c"] += perda_c.item() * b;   ac["w"] += w_dist * b
            ac["adv"] += perda_adv.item() * b
            ac["img"] += perda_img.item() * b; ac["msg"] += perda_msg.item() * b
            ac["acc"] += base.acuracia_bits(logits, mensagem) * b
            ac["n"] += b

        n = ac["n"]
        print(f"[treino] época {epoca:3d}  perda_c={ac['c']/n:+.3f}  W={ac['w']/n:+.3f}  "
              f"adv={ac['adv']/n:+.3f}  img_mse={ac['img']/n:.5f}  "
              f"msg_bce={ac['msg']/n:.4f}  acuracia_bits={ac['acc']/n:.4f}")

        # --- validação (reusa a métrica da Fase 0) + amostras ---
        val = base.avaliar(codificador, decodificador, loader_val, cfg)
        ok = "OK" if (val["acuracia_bits"] >= cfg["alvo_bitacc"] and val["psnr"] >= 35.0) else "  "
        print(f"[  val ] época {epoca:3d}  acuracia_bits={val['acuracia_bits']:.4f}  "
              f"psnr={val['psnr']:.2f}dB  ssim={val['ssim']:.4f}  linf={val['linf']:.4f}  {ok}")

        score = (val["psnr"] if val["acuracia_bits"] >= cfg["alvo_bitacc"]
                 else val["acuracia_bits"] - 100.0)
        if score > melhor["score"]:
            melhor = {"score": score, "epoca": epoca,
                      **{k: val[k] for k in ("acuracia_bits", "psnr", "ssim")}}
            torch.save({"codificador": codificador.state_dict(),
                        "decodificador": decodificador.state_dict(),
                        "critico": critico.state_dict(),
                        "epoca": epoca, "cfg": cfg, "val": val},
                       os.path.join(cfg["pasta_saida"], "ckpt_best.pt"))

        if epoca % cfg["salvar_cada"] == 0:
            base.salvar_amostras(codificador, loader_val, cfg, epoca)
            torch.save({"codificador": codificador.state_dict(),
                        "decodificador": decodificador.state_dict(),
                        "critico": critico.state_dict(),
                        "epoca": epoca, "cfg": cfg},
                       os.path.join(cfg["pasta_saida"], "ckpt_last.pt"))

    print(f"\n[fim] checkpoints e amostras em: {cfg['pasta_saida']}")
    print(f"[melhor] época {melhor['epoca']}: acuracia_bits={melhor['acuracia_bits']:.4f}  "
          f"psnr={melhor['psnr']:.2f}dB  ssim={melhor['ssim']:.4f}")
    return {"acuracia_bits": melhor["acuracia_bits"], "psnr": melhor["psnr"], "ssim": melhor["ssim"]}


if __name__ == "__main__":
    treinar()


In [ ]:
%%writefile fase3_esteganalisador.py
"""
Fase 3 — Indetectabilidade dirigida: SRNet como 2º adversário.

O que muda em relação à Fase 2:
  - Entra um SEGUNDO adversário, a SRNet (Boroumand, Chen e Fridrich, 2019). Os dois
    adversários cobram coisas diferentes do gerador e por isso convivem:
      * Crítico WGAN-GP -> REALISMO. Score escalar de "quão natural é esta imagem".
      * SRNet           -> INDETECTABILIDADE. Classificador portadora vs. estego, treinado
                           para achar o micro-ruído residual que a ocultação deixa.
    Uma imagem pode ser perfeitamente realista e ainda assim ter assinatura estatística
    detectável — é exatamente essa lacuna que a SRNet fecha.
  - A perda do gerador ganha um 4º termo: enganar a SRNet (fazer a estego ser classificada
    como portadora).

Por que a SRNet e não outro detector:
  - Ela não usa filtro passa-alta fixo: aprende o residual sozinha.
  - As 7 primeiras camadas NÃO têm pooling, de propósito. Pooling é filtro passa-baixa e
    destruiria justamente o sinal de alta frequência que denuncia a esteganografia. A redução
    de resolução só começa na camada 8.

Warm start: gerador e crítico vêm da Fase 2 (ckpt_best). A SRNet começa DO ZERO — um detector
pré-treinado contra outro gerador entra viciado e mede a coisa errada.

CRITÉRIOS DE SAÍDA da Fase 3 (os três precisam valer):
  (1) recuperação:        acuracia_bits(val) >= 0.99   (não regredir o que a Fase 1 conquistou)
  (2) imperceptibilidade: PSNR(val) não cair mais que ~1 dB em relação aos 36,30 dB da Fase 2
  (3) o detector do laço estacionar perto do acaso (AUC ~ 0.5)

>>> LEIA ISTO ANTES DE COMEMORAR O CRITÉRIO (3) <<<
A AUC reportada aqui é a do detector que PARTICIPOU DO TREINO. Ela pode estar em 0,5 porque o
gerador ficou bom OU porque aquela instância de SRNet não convergiu — os dois casos produzem o
mesmo número. Uma AUC de 0,5 no laço NÃO é evidência de segurança; é evidência sobre aquele
adversário específico. A prova real é a Fase 4, que treina um esteganalisador independente e
mede P_E. Ver ROADMAP.md.
E lembre da simetria: AUC abaixo de 0,5 não é "melhor que aleatório". Significa que o detector
aprendeu a separação com os rótulos invertidos, e basta inverter a regra de decisão. Por isso
o log imprime também max(AUC, 1-AUC).

O QUE O GERADOR PEDE À SRNet (cfg["alvo_srnet"]) — importa mais do que parece:
  - "portadora" (alvo 0) é o objetivo clássico de "enganar o classificador". Ele NÃO tem mínimo
    no equilíbrio: a perda decresce monotonicamente conforme o logit vai a -infinito, então o
    ótimo é "mais portadora que as portadoras". O gerador passa do ponto e a AUC despenca ABAIXO
    de 0,5 — as imagens voltam a ser separáveis, agora com o sinal trocado.
  - "equilibrio" (alvo suave 0,5) tem mínimo EXATO em probabilidade 0,5, que é literalmente
    "o detector não consegue decidir". É o default, e é o que a Fase 3 quer.
  Com "equilibrio", a coluna `enganar` passa a ter piso em ln 2 ~ 0,693 em vez de 0: 0,693 é o
  equilíbrio perfeito, e valores acima indicam separação — em qualquer das duas direções.

Reaproveita: Codificador/Decodificador/loaders/métricas da Fase 0; Crítico e penalidade de
gradiente da Fase 2.

Rodar:  python3.10 fase3_esteganalisador.py
"""
import os
import csv
import time
import random
import shutil
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F

import fase0_codificador_decodificador as base    # Codificador, Decodificador, loaders, métricas
import fase2_wgan_gp as fase2            # Critico, penalidade_gradiente


# ---------------------------------------------------------------------------
# CONFIG — herda TODAS as chaves da Fase 2 (que já herda as da Fase 0).
# ---------------------------------------------------------------------------
CONFIG = dict(fase2.CONFIG)
CONFIG.update({
    # --- 2º adversário (SRNet) ---
    # Limiar do critério (2). A Fase 2 fechou em 36,30 dB; o ROADMAP admite cair no máximo
    # ~1 dB. Fica em cfg para que a marca por época e o veredito final usem O MESMO número —
    # antes a marca usava 35,0 e o veredito 35,3, e épocas como 35,06/35,22 apareciam
    # aprovadas no log e reprovavam no fim.
    "alvo_psnr": 35.3,

    "lambda_srnet": 0.01,      # peso do termo "enganar a SRNet" no gerador. É O NOVO KNOB.
                               # Mesmo trade-off do lambda_adv: alto demais derruba a acurácia
                               # de bits, baixo demais não muda nada. Comece em 0,01 e suba.
    "n_srnet": 1,              # passos da SRNet por passo do gerador. Custa ~+22% no tempo por
                               # passo (o gargalo é o gerador denso, não a SRNet). Suba para 2
                               # se a srnet_bce ficar travada em ~0,69 (detector não aprendeu).
    "alvo_srnet": "equilibrio", # O QUE o gerador pede à SRNet. Ver nota no topo do arquivo.
                               #  "equilibrio" -> alvo suave 0,5: mínimo exato em "o detector não
                               #                  consegue decidir". É o que a Fase 3 quer.
                               #  "portadora"  -> alvo 0: decresce sem parar, o gerador passa do
                               #                  ponto e a AUC cai ABAIXO de 0,5. Só para comparar.
    "lr_srnet": 1e-4,          # learning rate da SRNet
    "srnet_compacta": False,   # True = metade dos filtros. Use para uma primeira rodada rápida
                               # de sanidade; os resultados finais devem ser com a SRNet cheia.

    # --- treino ---
    "epocas": 60,
    "usar_amp": False,         # a GP do crítico continua no laço; ver docstring da Fase 2
    "carregar_fase0": None,    # superado: o warm start agora vem da Fase 2
    "carregar_fase2": "runs/phase2_bossbase/ckpt_best.pt",
    "pasta_saida": "runs/phase3_bossbase",

    # --- persistência ---
    # Espelha ckpt_best.pt para fora do ambiente efêmero a cada melhoria. No Colab, aponte
    # para o Drive montado. Foi a ausência disto que fez os pesos das Fases 0 e 2 se perderem.
    "espelho_ckpt": None,      # ex.: "/content/drive/MyDrive/IC_stego/phase3"
})

# Preset GPU (Colab): junta o preset da Fase 2 com os ajustes do 2º adversário.
COLAB_GPU = dict(fase2.COLAB_GPU)
COLAB_GPU.update({
    "epocas": 60,
    "lambda_srnet": 0.01,
    "usar_amp": False,
    # Detecta as CPUs da máquina em vez de fixar um número: o T4 do Colab tem 2 vCPU e a L4
    # tem 8. Fixar 4 gera aviso no T4 e desperdiça a L4.
    "num_trabalhadores": max(1, min(4, (os.cpu_count() or 2) - 1)),
})


# ---------------------------------------------------------------------------
# SRNet — esteganalisador profundo (Boroumand, Chen e Fridrich, 2019)
#
# Quatro tipos de camada, nesta ordem:
#   Tipo 1 (x2)  conv-BN-ReLU                       -> extrai o residual, sem reduzir resolução
#   Tipo 2 (x5)  residual: conv-BN-ReLU-conv-BN + atalho
#   Tipo 3 (x4)  residual com redução: mesmo ramo + avg-pool 3x3 stride 2; atalho 1x1 stride 2
#   Tipo 4 (x1)  conv-BN + global average pooling   -> vetor de estatísticas da imagem
# Depois: uma camada linear -> 1 logit (portadora vs. estego).
# ---------------------------------------------------------------------------
class _Tipo1(nn.Module):
    def __init__(self, entrada, saida):
        super().__init__()
        self.conv = nn.Conv2d(entrada, saida, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(saida)

    def forward(self, x):
        return F.relu(self.bn(self.conv(x)), inplace=True)


class _Tipo2(nn.Module):
    """Residual sem redução de resolução: saída = x + F(x)."""
    def __init__(self, canais):
        super().__init__()
        self.c1 = nn.Conv2d(canais, canais, 3, padding=1, bias=False)
        self.b1 = nn.BatchNorm2d(canais)
        self.c2 = nn.Conv2d(canais, canais, 3, padding=1, bias=False)
        self.b2 = nn.BatchNorm2d(canais)

    def forward(self, x):
        h = F.relu(self.b1(self.c1(x)), inplace=True)
        h = self.b2(self.c2(h))
        return x + h


class _Tipo3(nn.Module):
    """Residual COM redução: ramo principal termina em avg-pool; atalho é conv 1x1 stride 2."""
    def __init__(self, entrada, saida):
        super().__init__()
        self.c1 = nn.Conv2d(entrada, saida, 3, padding=1, bias=False)
        self.b1 = nn.BatchNorm2d(saida)
        self.c2 = nn.Conv2d(saida, saida, 3, padding=1, bias=False)
        self.b2 = nn.BatchNorm2d(saida)
        self.pool = nn.AvgPool2d(3, stride=2, padding=1)
        self.atalho_conv = nn.Conv2d(entrada, saida, 1, stride=2, bias=False)
        self.atalho_bn = nn.BatchNorm2d(saida)

    def forward(self, x):
        h = F.relu(self.b1(self.c1(x)), inplace=True)
        h = self.pool(self.b2(self.c2(h)))
        return self.atalho_bn(self.atalho_conv(x)) + h


class _Tipo4(nn.Module):
    """Última convolução + global average pooling -> (B, canais)."""
    def __init__(self, entrada, saida):
        super().__init__()
        self.c = nn.Conv2d(entrada, saida, 3, padding=1, bias=False)
        self.b = nn.BatchNorm2d(saida)

    def forward(self, x):
        h = F.relu(self.b(self.c(x)), inplace=True)
        return h.mean(dim=(2, 3))          # média espacial global


class SRNet(nn.Module):
    """Imagem -> 1 logit. Alto = "tem mensagem escondida" (estego)."""
    def __init__(self, canais, compacta=False):
        super().__init__()
        e = 0.5 if compacta else 1.0
        f = lambda n: max(4, int(n * e))

        self.tipo1 = nn.Sequential(_Tipo1(canais, f(64)), _Tipo1(f(64), f(16)))
        self.tipo2 = nn.Sequential(*[_Tipo2(f(16)) for _ in range(5)])
        self.tipo3 = nn.Sequential(
            _Tipo3(f(16), f(16)),
            _Tipo3(f(16), f(64)),
            _Tipo3(f(64), f(128)),
            _Tipo3(f(128), f(256)),
        )
        self.tipo4 = _Tipo4(f(256), f(512))
        self.para_logit = nn.Linear(f(512), 1)

    def forward(self, x):
        h = self.tipo1(x)      # sem pooling: preserva o residual de alta frequência
        h = self.tipo2(h)      # ainda sem pooling
        h = self.tipo3(h)      # a partir daqui, reduz resolução
        h = self.tipo4(h)      # (B, C)
        return self.para_logit(h).squeeze(1)   # (B,)


# ---------------------------------------------------------------------------
# Métrica de detecção
# ---------------------------------------------------------------------------
def auc_binaria(scores, rotulos):
    """AUC pela estatística de Mann-Whitney (equivale à área sob a curva ROC).

    scores  : tensor (N,) — saída contínua do detector
    rotulos : tensor (N,) — 1 = estego, 0 = portadora

    Interpretação: 0,5 = classificador aleatório. Independe do limiar de decisão, e é por
    isso que ela vale mais do que a acurácia para julgar um detector.
    """
    scores = scores.detach().float().flatten()
    rotulos = rotulos.detach().float().flatten()
    n_pos = int(rotulos.sum().item())
    n_neg = int(rotulos.numel() - n_pos)
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    ordem = torch.argsort(scores)
    postos = torch.empty_like(scores)
    postos[ordem] = torch.arange(1, scores.numel() + 1, dtype=scores.dtype, device=scores.device)
    soma_postos_pos = postos[rotulos == 1].sum().item()
    return (soma_postos_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)


@torch.no_grad()
def avaliar_srnet(codificador, srnet, loader, cfg):
    """Roda a SRNet sobre pares (portadora, estego) da validação e devolve AUC e acurácia."""
    dispositivo = cfg["dispositivo"]
    codificador.eval(); srnet.eval()
    todos_scores, todos_rotulos = [], []

    for portadora in loader:
        portadora = portadora.to(dispositivo, non_blocking=True)
        b = portadora.size(0)
        mensagem = base.mensagem_aleatoria(b, cfg)
        estego = codificador(portadora, base.espalhar_mensagem(mensagem, cfg))

        logits = srnet(torch.cat([portadora, estego], dim=0)).cpu()   # lote único, como no treino
        todos_scores.append(logits)
        todos_rotulos.append(torch.cat([torch.zeros(b), torch.ones(b)]))

    scores = torch.cat(todos_scores)
    rotulos = torch.cat(todos_rotulos)
    auc = auc_binaria(scores, rotulos)
    acc = ((scores > 0).float() == rotulos).float().mean().item()
    return {"auc": auc, "acuracia_deteccao": acc}


# ---------------------------------------------------------------------------
# Warm start
# ---------------------------------------------------------------------------
def _carregar_gerador(caminho, codificador, decodificador, critico, dispositivo, log=print):
    """Carrega pesos da Fase 2. Tolera o nome antigo das chaves (encoder/decoder)."""
    if not caminho or not os.path.exists(caminho):
        log(f"[warm start] {caminho!r} não encontrado -> gerador DO ZERO.\n"
              "             A Fase 3 é para ser construída SOBRE a Fase 2. Rode a Fase 2 antes,\n"
              "             ou aponte 'carregar_fase2' para o ckpt_best.pt no Drive.")
        return False

    estado = torch.load(caminho, map_location=dispositivo)
    cod = estado.get("codificador", estado.get("encoder"))
    dec = estado.get("decodificador", estado.get("decoder"))
    if cod is None or dec is None:
        raise KeyError(f"{caminho}: não achei os pesos do codificador/decodificador. "
                       f"Chaves presentes: {list(estado.keys())}")
    codificador.load_state_dict(cod)
    decodificador.load_state_dict(dec)

    if "critico" in estado:
        critico.load_state_dict(estado["critico"])
        log(f"[warm start] gerador + crítico carregados de {caminho} (época {estado.get('epoca', '?')})")
    else:
        log(f"[warm start] gerador carregado de {caminho}; crítico DO ZERO "
              "(checkpoint não tinha os pesos do crítico)")
    return True


class _Log:
    """Imprime na tela E grava em arquivo, com flush a cada linha.

    O flush importa: se a sessão do Colab cair no meio do treino, o que já foi impresso
    continua no disco. Sem ele, o buffer se perde junto com o runtime.
    """
    def __init__(self, caminho):
        self.caminho = caminho
        self.arquivo = open(caminho, "a", encoding="utf-8")

    def __call__(self, texto=""):
        print(texto)
        self.arquivo.write(texto + "\n")
        self.arquivo.flush()

    def fechar(self):
        self.arquivo.close()


def _espelhar(cfg, origem, log=print):
    """Copia o checkpoint para fora do ambiente efêmero (Drive), se configurado."""
    destino = cfg.get("espelho_ckpt")
    if not destino:
        return
    # Se o espelho aponta para a própria pasta_saida (caso comum quando se roda direto no
    # Drive montado), copiar é um no-op que só levanta SameFileError a cada época. Silencia
    # sem mascarar falhas reais de escrita.
    if os.path.abspath(destino) == os.path.abspath(cfg["pasta_saida"]):
        return
    try:
        os.makedirs(destino, exist_ok=True)
        shutil.copy2(origem, os.path.join(destino, os.path.basename(origem)))
    except Exception as e:
        log(f"[espelho] falhou ao copiar para {destino}: {e}")


# ---------------------------------------------------------------------------
# Treino com DOIS adversários
# ---------------------------------------------------------------------------
def treinar():
    cfg = CONFIG
    torch.manual_seed(cfg["semente"]); random.seed(cfg["semente"])
    os.makedirs(cfg["pasta_saida"], exist_ok=True)
    dispositivo = cfg["dispositivo"]
    tm = base._tamanho_msg(cfg)
    bpp = cfg["profundidade_dados"] * (tm / cfg["tamanho_img"]) ** 2

    # Tudo que for impresso daqui em diante também vai para treino.log.
    log = _Log(os.path.join(cfg["pasta_saida"], "treino.log"))
    log(f"===== Fase 3 iniciada em {datetime.now():%Y-%m-%d %H:%M:%S} =====")
    log("[cfg] " + "  ".join(f"{k}={cfg[k]}" for k in
        ("tamanho_img", "tamanho_msg", "tamanho_lote", "largura", "epocas",
         "lambda_imagem", "lambda_mensagem", "lambda_residuo", "lambda_adv",
         "lambda_srnet", "n_critic", "n_srnet", "lr_srnet", "srnet_compacta")))
    log(f"[infra] dispositivo={dispositivo}  payload={bpp:.4g} bpp  "
          f"lambda_adv={cfg['lambda_adv']}  lambda_srnet={cfg['lambda_srnet']}  "
          f"n_critic={cfg['n_critic']}  n_srnet={cfg['n_srnet']}"
          + ("  [SRNet COMPACTA]" if cfg["srnet_compacta"] else ""))

    loader_treino, loader_val = base.montar_loaders(cfg)

    codificador = base.Codificador(cfg["canais"], cfg["profundidade_dados"], cfg["largura"]).to(dispositivo)
    decodificador = base.Decodificador(cfg["canais"], cfg["profundidade_dados"], cfg["largura"]).to(dispositivo)
    critico = fase2.Critico(cfg["canais"], cfg["largura_critico"]).to(dispositivo)
    srnet = SRNet(cfg["canais"], compacta=cfg["srnet_compacta"]).to(dispositivo)

    _carregar_gerador(cfg.get("carregar_fase2"), codificador, decodificador, critico,
                      dispositivo, log=log)
    n_par = sum(p.numel() for p in srnet.parameters())
    log(f"[srnet] iniciada DO ZERO — {n_par/1e6:.2f}M parâmetros")

    opt_g = torch.optim.Adam(list(codificador.parameters()) + list(decodificador.parameters()),
                             lr=cfg["taxa_aprend"], betas=(cfg["beta1"], cfg["beta2"]))
    opt_c = torch.optim.Adam(critico.parameters(),
                             lr=cfg["lr_critico"], betas=(cfg["beta1"], cfg["beta2"]))
    opt_s = torch.optim.Adam(srnet.parameters(), lr=cfg["lr_srnet"])   # betas padrão: é classificador, não crítico

    caminho_csv = os.path.join(cfg["pasta_saida"], "metricas_val.csv")
    with open(caminho_csv, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(["epoca", "acuracia_bits", "psnr", "ssim", "auc_srnet", "w_dist"])

    # Diagnóstico do laço de treino. São estas colunas que dizem SE o treino está saudável —
    # 'srnet_bce' travada em ~0,69 = detector não aprendeu; 'enganar' crescendo sem parar = o
    # termo da SRNet está dominando a perda do gerador.
    caminho_csv_treino = os.path.join(cfg["pasta_saida"], "metricas_treino.csv")
    with open(caminho_csv_treino, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(["epoca", "perda_c", "w_dist", "adv", "srnet_bce", "enganar",
                                "img_mse", "msg_bce", "acuracia_bits"])

    melhor = {"score": -1.0, "epoca": 0, "acuracia_bits": 0.0, "psnr": 0.0, "ssim": 0.0, "auc": float("nan")}

    t_inicio = time.time()
    for epoca in range(1, cfg["epocas"] + 1):
        t_epoca = time.time()
        codificador.train(); decodificador.train(); critico.train(); srnet.train()
        ac = {"c": 0.0, "w": 0.0, "adv": 0.0, "s": 0.0, "eng": 0.0,
              "img": 0.0, "msg": 0.0, "acc": 0.0, "n": 0}

        for portadora in loader_treino:
            portadora = portadora.to(dispositivo, non_blocking=True)
            b = portadora.size(0)

            # ============ 1) CRÍTICO (realismo), n_critic vezes ============
            for _ in range(cfg["n_critic"]):
                mensagem = base.mensagem_aleatoria(b, cfg)
                with torch.no_grad():
                    estego = codificador(portadora, base.espalhar_mensagem(mensagem, cfg))
                score_real = critico(portadora).mean()
                score_falsa = critico(estego).mean()
                gp = fase2.penalidade_gradiente(critico, portadora, estego, dispositivo)
                perda_c = score_falsa - score_real + cfg["lambda_gp"] * gp
                opt_c.zero_grad(); perda_c.backward(); opt_c.step()

            w_dist = (score_real - score_falsa).item()

            # ============ 2) SRNet (detecção), n_srnet vezes ============
            # Classificador binário puro: portadora = 0, estego = 1.
            # Lote ÚNICO com portadora e estego juntas. A SRNet tem BatchNorm em todas as
            # camadas; se as duas classes vierem em passagens separadas, cada lote é normalizado
            # pelas próprias estatísticas e a rede pode separar pela composição do lote em vez
            # do conteúdo — um viés que só aparece depois, em eval(). Concatenar elimina isso.
            for _ in range(cfg["n_srnet"]):
                mensagem = base.mensagem_aleatoria(b, cfg)
                with torch.no_grad():
                    estego = codificador(portadora, base.espalhar_mensagem(mensagem, cfg))
                logits_s = srnet(torch.cat([portadora, estego], dim=0))
                alvos_s = torch.cat([torch.zeros(b, device=dispositivo),
                                     torch.ones(b, device=dispositivo)])
                perda_s = F.binary_cross_entropy_with_logits(logits_s, alvos_s)
                opt_s.zero_grad(); perda_s.backward(); opt_s.step()

            # ============ 3) GERADOR, 1 vez, contra os DOIS adversários ============
            mensagem = base.mensagem_aleatoria(b, cfg)
            estego = codificador(portadora, base.espalhar_mensagem(mensagem, cfg))
            logits = base.agregar_logits(decodificador(estego), cfg)

            perda_img = F.mse_loss(estego, portadora)
            perda_msg = F.binary_cross_entropy_with_logits(logits, mensagem)
            perda_res = (estego - portadora).abs().mean()
            perda_adv = -critico(estego).mean()          # quer SUBIR o score de realismo

            # Mesma composição de lote do passo da SRNet, para a BatchNorm ver as mesmas
            # estatísticas; o gradiente só flui pela metade estego.
            logits_g = srnet(torch.cat([portadora.detach(), estego], dim=0))
            logit_est_g = logits_g[b:]
            valor_alvo = 0.5 if cfg["alvo_srnet"] == "equilibrio" else 0.0
            perda_eng = F.binary_cross_entropy_with_logits(
                logit_est_g, torch.full_like(logit_est_g, valor_alvo))

            perda_g = (cfg["lambda_imagem"] * perda_img
                       + cfg["lambda_mensagem"] * perda_msg
                       + cfg["lambda_residuo"] * perda_res
                       + cfg["lambda_adv"] * perda_adv
                       + cfg["lambda_srnet"] * perda_eng)
            opt_g.zero_grad(); perda_g.backward(); opt_g.step()

            ac["c"] += perda_c.item() * b;    ac["w"] += w_dist * b
            ac["adv"] += perda_adv.item() * b; ac["s"] += perda_s.item() * b
            ac["eng"] += perda_eng.item() * b
            ac["img"] += perda_img.item() * b; ac["msg"] += perda_msg.item() * b
            ac["acc"] += base.acuracia_bits(logits, mensagem) * b
            ac["n"] += b

        n = ac["n"]
        seg = time.time() - t_epoca
        restam = (cfg["epocas"] - epoca) * (time.time() - t_inicio) / epoca
        log(f"[tempo ] época {epoca:3d}  {seg:6.1f}s  ({ac['n']//cfg['tamanho_lote']} passos de gerador)"
            f"  restam ~{restam/60:.0f} min")
        log(f"[treino] época {epoca:3d}  perda_c={ac['c']/n:+.3f}  W={ac['w']/n:+.3f}  "
              f"srnet_bce={ac['s']/n:.4f}  enganar={ac['eng']/n:.4f}  "
              f"img_mse={ac['img']/n:.5f}  msg_bce={ac['msg']/n:.4f}  "
              f"acuracia_bits={ac['acc']/n:.4f}")

        # --- validação: qualidade (Fase 0) + detecção (nova) ---
        val = base.avaliar(codificador, decodificador, loader_val, cfg)
        det = avaliar_srnet(codificador, srnet, loader_val, cfg)
        auc, auc_ef = det["auc"], max(det["auc"], 1.0 - det["auc"])

        c1 = val["acuracia_bits"] >= cfg["alvo_bitacc"]
        c2 = val["psnr"] >= cfg["alvo_psnr"]
        c3 = abs(auc - 0.5) <= 0.05
        marca = "".join(["1" if c1 else ".", "2" if c2 else ".", "3" if c3 else "."])
        log(f"[  val ] época {epoca:3d}  acuracia_bits={val['acuracia_bits']:.4f}  "
              f"psnr={val['psnr']:.2f}dB  ssim={val['ssim']:.4f}  "
              f"auc_srnet={auc:.4f} (efetiva {auc_ef:.4f})  criterios[{marca}]")

        with open(caminho_csv, "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([epoca, f"{val['acuracia_bits']:.4f}", f"{val['psnr']:.2f}",
                                    f"{val['ssim']:.4f}", f"{auc:.4f}", f"{ac['w']/n:.4f}"])
        with open(caminho_csv_treino, "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([epoca, f"{ac['c']/n:.4f}", f"{ac['w']/n:.4f}",
                                    f"{ac['adv']/n:.4f}", f"{ac['s']/n:.4f}", f"{ac['eng']/n:.4f}",
                                    f"{ac['img']/n:.6f}", f"{ac['msg']/n:.4f}", f"{ac['acc']/n:.4f}"])

        # Espelha os registros TODA época — não só quando o checkpoint melhora. Se a sessão
        # cair na época 47, o log e as métricas até ali já estão salvos fora do runtime.
        for reg in (caminho_csv, caminho_csv_treino, os.path.join(cfg["pasta_saida"], "treino.log")):
            _espelhar(cfg, reg, log)

        # --- melhor checkpoint ---
        # Critério DELIBERADAMENTE igual ao da Fase 2 (PSNR com acurácia como porteira), e não
        # a AUC do laço. Selecionar pela AUC do detector que participou do treino é escolher o
        # checkpoint que melhor engana ESTE detector — exatamente o viés que a Fase 4 existe
        # para não cometer. A AUC fica no log como monitoramento, não como critério.
        score = (val["psnr"] if c1 else val["acuracia_bits"] - 100.0)
        if score > melhor["score"]:
            melhor = {"score": score, "epoca": epoca, "auc": auc,
                      **{k: val[k] for k in ("acuracia_bits", "psnr", "ssim")}}
            destino = os.path.join(cfg["pasta_saida"], "ckpt_best.pt")
            torch.save({"codificador": codificador.state_dict(),
                        "decodificador": decodificador.state_dict(),
                        "critico": critico.state_dict(),
                        "srnet": srnet.state_dict(),
                        "epoca": epoca, "cfg": cfg, "val": val, "det": det},
                       destino)
            _espelhar(cfg, destino, log)

        if epoca % cfg["salvar_cada"] == 0:
            base.salvar_amostras(codificador, loader_val, cfg, epoca)
            destino = os.path.join(cfg["pasta_saida"], "ckpt_last.pt")
            torch.save({"codificador": codificador.state_dict(),
                        "decodificador": decodificador.state_dict(),
                        "critico": critico.state_dict(),
                        "srnet": srnet.state_dict(),
                        "epoca": epoca, "cfg": cfg},
                       destino)
            _espelhar(cfg, destino, log)

    # --- fechamento ---
    auc_ef = max(melhor["auc"], 1.0 - melhor["auc"])
    log(f"\n[fim] checkpoints, amostras e metricas_val.csv em: {cfg['pasta_saida']}")
    log(f"[melhor] época {melhor['epoca']}: acuracia_bits={melhor['acuracia_bits']:.4f}  "
          f"psnr={melhor['psnr']:.2f}dB  ssim={melhor['ssim']:.4f}  auc_srnet={melhor['auc']:.4f}")
    log("\nCRITÉRIOS DE SAÍDA DA FASE 3:")
    log(f"  (1) acuracia_bits >= 0.99          -> {melhor['acuracia_bits']:.4f}  "
          f"{'OK' if melhor['acuracia_bits'] >= 0.99 else 'FALHOU: baixe lambda_srnet'}")
    log(f"  (2) PSNR >= {cfg['alvo_psnr']} dB (Fase 2 - 1dB) -> {melhor['psnr']:.2f}  "
          f"{'OK' if melhor['psnr'] >= cfg['alvo_psnr'] else 'FALHOU: baixe lambda_srnet ou suba lambda_imagem'}")
    if abs(melhor["auc"] - 0.5) <= 0.05:
        veredito = "OK"
    elif melhor["auc"] > 0.5:
        veredito = "detector separando: suba lambda_srnet"
    else:
        veredito = ("gerador PASSOU DO PONTO (estego com score menor que portadora). "
                    "NÃO suba lambda_srnet — isso piora. Use alvo_srnet='equilibrio'")
    log(f"  (3) AUC do laço ~ 0.5              -> {melhor['auc']:.4f} (efetiva {auc_ef:.4f})  {veredito}")
    log("\nATENÇÃO: o critério (3) mede o detector que participou do treino. Ele NÃO prova")
    log("segurança — pode significar que aquela SRNet não convergiu. A prova é a Fase 4,")
    log("que treina um esteganalisador independente e mede P_E. Ver ROADMAP.md.")
    log(f"\n[registros] treino.log, metricas_val.csv e metricas_treino.csv em {cfg['pasta_saida']}")
    if not cfg.get("espelho_ckpt"):
        log("\nLEMBRETE: 'espelho_ckpt' está vazio. Se isto rodou no Colab, BAIXE AGORA o")
        log("ckpt_best.pt E os arquivos de registro — foi assim que os pesos das Fases 0 e 2")
        log("se perderam, e a saída da célula do notebook some junto com o runtime.")
    else:
        _espelhar(cfg, os.path.join(cfg["pasta_saida"], "treino.log"), log)
        log(f"[espelho] registros e checkpoints copiados para {cfg['espelho_ckpt']}")
    log.fechar()
    return {"acuracia_bits": melhor["acuracia_bits"], "psnr": melhor["psnr"],
            "ssim": melhor["ssim"], "auc": melhor["auc"]}


if __name__ == "__main__":
    treinar()


## 4. Dataset (BOSSBase, grayscale)

Pula o download se já estiver no Drive.

In [ ]:
import os, subprocess

destino = os.path.join(PASTA_DADOS_BASE, 'BOSSbase_1.01')
if os.path.isdir(destino) and len(os.listdir(destino)) > 100:
    print('BOSSBase já está em', destino, '-', len(os.listdir(destino)), 'arquivos')
else:
    url = 'http://dde.binghamton.edu/download/ImageDB/BOSSbase_1.01.zip'
    print('Baixando BOSSBase (~2 min)...')
    subprocess.run(['wget', '-q', '--show-progress', '-O', '/content/boss.zip', url], check=True)
    subprocess.run(['unzip', '-q', '/content/boss.zip', '-d', PASTA_DADOS_BASE], check=True)
    print('Pronto:', len(os.listdir(destino)), 'imagens')


## 5. Localizar o checkpoint da Fase 2

A Fase 3 é construída **sobre** a Fase 2 — sem esse checkpoint ela treinaria do zero e não seria
Fase 3 coisa nenhuma. A célula procura no Drive e mostra o que achou.

In [ ]:
import glob, os, torch

candidatos = sorted(glob.glob(os.path.join(BASE, '**', 'ckpt_best.pt'), recursive=True))
print('Checkpoints encontrados no Drive:')
for c in candidatos:
    try:
        e = torch.load(c, map_location='cpu')
        ep = e.get('epoca', e.get('epoch', '?'))
        tem = [k for k in ('codificador', 'decodificador', 'critico', 'srnet') if k in e]
        val = e.get('val', {})
        extra = ('psnr=%.2f' % val['psnr']) if 'psnr' in val else ''
        print('  %-70s ép.%-5s %s %s' % (c, ep, tem, extra))
    except Exception as ex:
        print('  %-70s [erro ao ler: %s]' % (c, ex))

CKPT_FASE2 = os.path.join(PASTA_SAIDA_BASE, 'phase2_bossbase', 'ckpt_best.pt')
print('\nUsando:', CKPT_FASE2, '->', 'OK' if os.path.exists(CKPT_FASE2) else 'NÃO EXISTE — corrija acima')


## 6. Rodada de sanidade (3 épocas, SRNet compacta)

Serve só para ver se `lambda_srnet=0,01` não derruba a acurácia de bits. Melhor descobrir
em ~10 min do que depois de 4 horas. **Salva em pasta separada** para não sujar o run bom.

In [ ]:
import importlib, os
import fase0_codificador_decodificador, fase2_wgan_gp, fase3_esteganalisador as f3
for m in (fase0_codificador_decodificador, fase2_wgan_gp, f3):
    importlib.reload(m)

f3.CONFIG.update(f3.COLAB_GPU)
f3.CONFIG.update(
    pasta_dados     = os.path.join(PASTA_DADOS_BASE, 'BOSSbase_1.01'),
    canais          = 1,
    carregar_fase2  = CKPT_FASE2,
    pasta_saida     = os.path.join(PASTA_SAIDA_BASE, 'phase3_sanidade'),
    espelho_ckpt    = None,
    epocas          = 3,
    srnet_compacta  = True,
    max_imagens     = 2000,
)
_ = f3.treinar()


**Como ler o resultado acima**

Com `alvo_srnet='equilibrio'` (o default), a coluna `enganar` tem **piso em ln 2 ≈ 0,693**:
0,693 é o equilíbrio perfeito, e qualquer valor acima indica separação — nos dois sentidos.
A `auc_srnet` é o mesmo diagnóstico visto de outro ângulo.

| Sintoma | Causa provável | O que fazer |
|---|---|---|
| `acuracia_bits` despencou | `lambda_srnet` alto demais | baixe para 0,005 |
| `srnet_bce` travada em ~0,693 | SRNet não aprendeu nada | suba `lr_srnet` ou `n_srnet` |
| `auc_srnet` **acima** de 0,55 | detector separando | suba `lambda_srnet` |
| `auc_srnet` **abaixo** de 0,45 | gerador passou do ponto | **não** suba `lambda_srnet` — confira que `alvo_srnet='equilibrio'` |
| `enganar` subindo e se afastando de 0,693 | os dois casos acima | veja o sinal da AUC para saber qual |
| PSNR caiu muito | orçamento de distorção estourou | suba `lambda_imagem` |

Só siga para a célula 7 quando a acurácia de bits estiver ≥ 0,99 e a AUC não estiver fugindo
de 0,5 numa direção só.

## 6.5 Sonda de `lambda_srnet` — H1 vs H2

O run de 60 épocas (20/08) falhou o critério (3): AUC do laço cravada em **1,0000** em 55 de 60
épocas, mínima de 0,9978 na época 1. O PSNR caiu 1,8 dB e o SSIM 0,027 — o gerador pagou
qualidade e não comprou indetectabilidade nenhuma.

Duas hipóteses explicam isso, e elas pedem ações opostas:

- **H1 — o peso era baixo demais.** `perda_eng = BCE(z, 0.5)` cresce *linearmente* no logit
  (`enganar` chegou a 9,3, ou seja z ≈ 18,6), mas seu gradiente é `σ(z) − 0,5`, que **satura
  em 0,5**. Com `lambda_srnet = 0,01`, o empurrão efetivo é `0,01 × 0,5 = 0,005`. O termo
  *parecia* dominar a perda e na prática mal empurrava.
- **H2 — o jogo é invencível nesta configuração.** A SRNet re-vence a cada passo
  (`srnet_bce` colapsou para ~0,001 em 2 épocas) e nenhum peso resolve: o problema é o
  resíduo espalhado uniformemente a 0,0625 bpp.

O sinal a favor de H2 é `enganar` **subindo** de 1,95 para 9,3 — o detector fica mais
confiante ao longo do treino, o jogo diverge em vez de convergir.

**Regra de decisão.** Dois braços de 10 épocas, `lambda_srnet ∈ {0,5 · 2,0}`:

- Se a **AUC sair de 1,0000** em algum braço → **H1**: continue subindo `lambda_srnet` e rode
  as 60 épocas com o valor que mostrou movimento.
- Se a AUC ficar cravada em **1,0000 enquanto o PSNR desaba** → **H2**: pare de tunar. O
  caminho passa a ser reduzir payload (sonda com `tamanho_msg = 16` → 0,0156 bpp) ou atacar
  *onde* os bits vão, que é a Fase 6.

Custo: ~305 s/época × 10 × 2 braços ≈ **1 h 45**. Muito mais barato que repetir 5 h às cegas.


In [ ]:
import importlib, os, csv
import fase0_codificador_decodificador, fase2_wgan_gp, fase3_esteganalisador as f3

BRACOS  = [0.5, 2.0]
EPOCAS  = 10
resumo  = {}

for lam in BRACOS:
    for m in (fase0_codificador_decodificador, fase2_wgan_gp, f3):
        importlib.reload(m)

    saida = os.path.join(PASTA_SAIDA_BASE, f'phase3_sonda_lam{lam:g}')
    f3.CONFIG.update(f3.COLAB_GPU)
    f3.CONFIG.update(
        pasta_dados     = os.path.join(PASTA_DADOS_BASE, 'BOSSbase_1.01'),
        canais          = 1,
        carregar_fase2  = CKPT_FASE2,
        pasta_saida     = saida,
        espelho_ckpt    = saida,
        max_imagens     = None,
        epocas          = EPOCAS,
        srnet_compacta  = False,
        alvo_srnet      = 'equilibrio',
        lambda_srnet    = lam,
    )
    print('\n' + '=' * 78)
    print(f'BRAÇO lambda_srnet = {lam}   ->   {saida}')
    print('=' * 78)
    resumo[lam] = {'melhor': f3.treinar(), 'saida': saida}

print('\n' + '=' * 78)
print('SONDA — trajetória da AUC por época (o número que decide H1 vs H2)')
print('=' * 78)

BASE_AUC, BASE_PSNR = 1.0000, 36.25

for lam in BRACOS:
    caminho = os.path.join(resumo[lam]['saida'], 'metricas_val.csv')
    if not os.path.exists(caminho):
        print(f'\nlambda={lam}: metricas_val.csv não encontrado em {caminho}')
        continue
    with open(caminho, encoding='utf-8') as f:
        linhas = list(csv.DictReader(f))
    aucs  = [float(l['auc_srnet'])    for l in linhas]
    psnrs = [float(l['psnr'])         for l in linhas]
    accs  = [float(l['acuracia_bits'])for l in linhas]

    print(f'\n--- lambda_srnet = {lam} ---')
    print('  ép  ' + ' '.join(f'{int(l["epoca"]):6d}' for l in linhas))
    print('  AUC ' + ' '.join(f'{a:6.4f}' for a in aucs))
    print('  PSNR' + ' '.join(f'{p:6.2f}' for p in psnrs))
    print('  acc ' + ' '.join(f'{a:6.4f}' for a in accs))

    auc_min   = min(aucs)
    saiu      = auc_min < 0.9990
    psnr_fim  = sum(psnrs[-3:]) / len(psnrs[-3:])
    queda     = BASE_PSNR - psnr_fim
    resumo[lam].update(auc_min=auc_min, saiu=saiu, psnr_fim=psnr_fim, queda=queda,
                       acc_min=min(accs))
    print(f'  -> AUC mínima {auc_min:.4f} | PSNR médio das 3 últimas {psnr_fim:.2f} dB '
          f'(queda de {queda:+.2f} dB vs 36,25) | acc mínima {min(accs):.4f}')

print('\n' + '=' * 78)
moveu = [l for l in BRACOS if resumo.get(l, {}).get('saiu')]
if moveu:
    melhor_lam = min(moveu, key=lambda l: resumo[l]['auc_min'])
    print(f'H1 SUSTENTADA: a AUC saiu de 1,0 com lambda_srnet={melhor_lam} '
          f'(mínima {resumo[melhor_lam]["auc_min"]:.4f}).')
    print(f'Próximo passo: 60 épocas com lambda_srnet={melhor_lam}, vigiando o critério (2) —')
    print(f'o PSNR neste braço já caiu {resumo[melhor_lam]["queda"]:+.2f} dB em apenas {EPOCAS} épocas.')
    print('Se o PSNR furar 35,3 antes de a AUC chegar perto de 0,5, suba lambda_imagem junto.')
else:
    piores = ', '.join(f'lambda={l}: PSNR {resumo[l]["psnr_fim"]:.2f} dB '
                       f'({resumo[l]["queda"]:+.2f})' for l in BRACOS if l in resumo)
    print('H2 SUSTENTADA: a AUC ficou cravada em ~1,0 nos dois braços.')
    print(f'  {piores}')
    print('Pare de tunar lambda_srnet — o peso não é o gargalo. Duas saídas, nesta ordem:')
    print('  (a) sonda de payload: tamanho_msg=16 (0,0156 bpp), 10 épocas. Se a AUC continuar')
    print('      em 1,0 mesmo a 1/4 do payload, o problema é a ARQUITETURA, não a carga.')
    print('  (b) Fase 6 (carga adaptativa por textura): decidir ONDE os bits vão, em vez de')
    print('      espalhá-los uniformemente. É a limitação que o artigo do SBSeg já registrava.')
print('Em qualquer dos casos, a Fase 4 (P_E independente) segue sendo o instrumento de medida.')
print('=' * 78)


## 7. Treino completo (60 épocas, dataset inteiro)

~296 passos de gerador por época contra os 59 da sanidade — **100x mais atualizações no total**.
É disso que o gerador precisa para sair do mínimo da Fase 2 e reaprender a se esconder.

Olhe a linha `[tempo ]` da primeira época: ela já dá o ETA do run inteiro. Se passar de ~8h,
reduza para 40 épocas. O `ckpt_best.pt` é espelhado no Drive **a cada melhoria**, então mesmo
que a sessão morra na época 40 você fica com o melhor até ali.

Ligue a execução em segundo plano antes de rodar.

In [ ]:
import importlib, os
import fase0_codificador_decodificador, fase2_wgan_gp, fase3_esteganalisador as f3
for m in (fase0_codificador_decodificador, fase2_wgan_gp, f3):
    importlib.reload(m)

f3.CONFIG.update(f3.COLAB_GPU)
f3.CONFIG.update(
    pasta_dados     = os.path.join(PASTA_DADOS_BASE, 'BOSSbase_1.01'),
    canais          = 1,
    carregar_fase2  = CKPT_FASE2,
    pasta_saida     = os.path.join(PASTA_SAIDA_BASE, 'phase3_bossbase'),
    espelho_ckpt    = os.path.join(PASTA_SAIDA_BASE, 'phase3_bossbase'),
    max_imagens     = None,
    epocas          = 60,
    srnet_compacta  = False,
    lambda_srnet    = 0.01,
    alvo_srnet      = 'equilibrio',
)
resultado_p3 = f3.treinar()
print('\nRESULTADO FASE 3:', resultado_p3)


## 8. Amostras (portadora / estego / resíduo)

In [ ]:
import glob, os
from IPython.display import Image as IPyImage, display
amostras = sorted(glob.glob(os.path.join(PASTA_SAIDA_BASE, 'phase3_bossbase', 'samples_epoch*.png')))
if amostras:
    print('Última amostra:', amostras[-1], ' (linha 1=portadora, 2=estego, 3=resíduo)')
    display(IPyImage(filename=amostras[-1]))
else:
    print('Nenhuma amostra ainda — rode o treino primeiro.')


## 9. Curvas: qualidade, detecção e saúde do treino

Lidas de `metricas_val.csv` e `metricas_treino.csv` — os arquivos, não a saída da célula.
Isso significa que você pode gerar o gráfico numa sessão nova, depois de o runtime ter morrido.

In [ ]:
import csv, os
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image as IPyImage, display

pasta = os.path.join(PASTA_SAIDA_BASE, 'phase3_bossbase')

def coluna(arquivo, *nomes):
    saida = {n: [] for n in nomes}
    with open(os.path.join(pasta, arquivo)) as f:
        for r in csv.DictReader(f):
            for n in nomes:
                saida[n].append(float(r[n]))
    return saida

v = coluna('metricas_val.csv', 'epoca', 'psnr', 'auc_srnet', 'acuracia_bits')
t = coluna('metricas_treino.csv', 'epoca', 'srnet_bce', 'enganar')

fig, (a1, a2, a3) = plt.subplots(3, 1, figsize=(7.5, 7.5), sharex=True, dpi=140,
                                 gridspec_kw={'height_ratios': [1.3, 1, 1]})
a1.plot(v['epoca'], v['psnr'], color='#2a78d6', label='PSNR')
a1.axhline(35.3, ls='--', lw=1, color='#eb6834', label='piso (Fase 2 - 1 dB)')
a1.set_ylabel('PSNR (dB)'); a1.grid(alpha=.3); a1.legend(fontsize=8)
a1.set_title('Fase 3 — qualidade, detecção e saúde do treino')

a2.plot(v['epoca'], v['auc_srnet'], color='#0ca30c')
a2.axhline(0.5, ls='--', lw=1, color='#898781')
a2.axhspan(0.45, 0.55, color='#0ca30c', alpha=.08)
a2.set_ylabel('AUC (laço)'); a2.set_ylim(0, 1); a2.grid(alpha=.3)

a3.plot(t['epoca'], t['srnet_bce'], color='#8b5cf6', label='srnet_bce (detector aprendendo?)')
a3.plot(t['epoca'], t['enganar'], color='#eb6834', label='enganar (termo dominando?)')
a3.axhline(0.693, ls=':', lw=1, color='#898781')
a3.text(v['epoca'][0], 0.72, 'ln 2 = piso de ambas', fontsize=7, color='#898781')
a3.set_ylabel('perda'); a3.set_xlabel('época'); a3.grid(alpha=.3); a3.legend(fontsize=8)
fig.tight_layout()
saida = os.path.join(pasta, 'curva_fase3.png')
fig.savefig(saida, facecolor='white')
print('Salvo em', saida)
display(IPyImage(filename=saida))


## 10. Antes de fechar a sessão

O treino grava tudo em disco e espelha no Drive **a cada época** — você não depende da
saída desta aba. Confirme que estes cinco estão lá:

- [ ] `ckpt_best.pt` — os pesos
- [ ] `treino.log` — a saída completa, linha a linha, com a config no cabeçalho
- [ ] `metricas_val.csv` — PSNR, SSIM, acurácia, AUC por época
- [ ] `metricas_treino.csv` — `srnet_bce`, `enganar` e os demais termos da perda
- [ ] `curva_fase3.png`

Próximo passo: **Fase 4** — esteganalisador independente e P_E. É ela que diz se a AUC ≈ 0,5
daqui significa alguma coisa.

In [ ]:
import os, glob
pasta = os.path.join(PASTA_SAIDA_BASE, 'phase3_bossbase')
for f in sorted(glob.glob(os.path.join(pasta, '*'))):
    print('%10.1f KB  %s' % (os.path.getsize(f) / 1024, os.path.basename(f)))
